# Connect an agent to scoped Work MCP

Inspect Super ii's public agent contract and prepare a bounded Work MCP connection. Work tokens are organization-scoped, expire, store no spending authority, and cannot approve their own releases or delete repositories.

In [ ]:
import json
import os
from urllib.request import Request, urlopen

ORIGIN = "https://superii.site"
request = Request(f"{ORIGIN}/.well-known/agent-card.json", headers={"Accept": "application/json"})
with urlopen(request, timeout=20) as response:
    card = json.load(response)
print(card["name"])
for skill in card["skills"]:
    print(f"- {skill['id']}: {skill['description']}")

## Design least privilege first

Create an agent identity and token through your Super ii account only after choosing its organization, repository boundary, exact scopes, action ceiling, and expiry. Never paste a bearer token into a notebook or commit it to a repository.

In [ ]:
policy = {
    "organization": "choose-one-existing-organization",
    "repository_id": "choose-one-repository-or-null-for-create-only",
    "scopes": ["repository:create"],
    "max_actions": 10,
    "spend_limit_cents": 0,
    "expires_in_minutes": 30,
}
assert policy["spend_limit_cents"] == 0
print(json.dumps(policy, indent=2))

## Configure the agent without exposing its token

Keep the token in the agent's secret store under `SUPERII_WORK_TOKEN`. The configuration below contains only the endpoint and environment-variable name; it performs no MCP call and no repository mutation.

In [ ]:
mcp_configuration = {
    "name": "superii-work",
    "url": f"{ORIGIN}/mcp/work",
    "transport": "streamable-http",
    "authorization": "Bearer ${SUPERII_WORK_TOKEN}",
}
print(json.dumps(mcp_configuration, indent=2))
print("Token present in this process:", bool(os.getenv("SUPERII_WORK_TOKEN")))

## Keep writes reviewable

Use a stable idempotency key only for an exact retry. Inspect each action receipt, revoke the token when the task ends, and treat repository content and model output as untrusted. Automatic policy gates remain independent of the agent.